# Μάθημα 13 - Μνήμη Αντιπροσώπου


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-5-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Τύποι Μνήμης Πράκτορα

Οι πράκτορες τεχνητής νοημοσύνης μπορούν να αξιοποιήσουν διάφορους τύπους μνήμης, ο καθένας εξυπηρετεί ένα διακριτό σκοπό:

### Εργαζόμενη Μνήμη
Το ίδιο το νήμα της συνομιλίας — τα μηνύματα που ανταλλάσσονται σε μια συνεδρία. Ο πράκτορας μπορεί να ανατρέξει σε προηγούμενα μηνύματα στο ίδιο νήμα για να διατηρήσει τη συνοχή. Στο MAF αυτό δημιουργείται μέσω της **`agent.create_session()`**, που επιστρέφει ένα `AgentSession`.

### Βραχυπρόθεσμη Μνήμη
Πληροφορίες που διαρκούν για τη διάρκεια μιας εργασίας ή συνεδρίας αλλά δεν αποθηκεύονται μόνιμα. Για παράδειγμα, ο πράκτορας μπορεί να συγκεντρώνει στοιχεία κατά τη διάρκεια μιας πολυ-γύρου συνομιλίας προγραμματισμού και να τα χρησιμοποιεί για να παράγει ένα τελικό δρομολόγιο.

### Μακροπρόθεσμη Μνήμη
Προτιμήσεις και στοιχεία που παραμένουν **διαχρονικά μεταξύ των συνεδριών**. Ένας επιστρέφων χρήστης δεν θα πρέπει να επαναλαμβάνει τους διατροφικούς περιορισμούς ή το στυλ ταξιδιού του. Η μακροπρόθεσμη μνήμη υποστηρίζεται συνήθως από ένα εξωτερικό αποθετήριο — μια βάση δεδομένων, αρχείο ή δείκτη διανυσμάτων — και εμφανίζεται στον πράκτορα μέσω εργαλείων.


## Εργαζόμενη Μνήμη με Συνεδρίες

Η απλούστερη μορφή μνήμης είναι η συνεδρία συνομιλίας. Όταν περνάτε την ίδια συνεδρία (δημιουργημένη μέσω `agent.create_session()`) σε διαδοχικές κλήσεις `agent.run()`, ο πράκτορας βλέπει ολόκληρο το ιστορικό της συνομιλίας και μπορεί να ανακαλέσει προηγούμενες λεπτομέρειες.

Ας δημιουργήσουμε έναν ταξιδιωτικό πράκτορα και να επιδείξουμε την εργαζόμενη μνήμη.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Ο πράκτορας ανακάλεσε σωστά τον προϋπολογισμό επειδή και τα δύο μηνύματα μοιράζονται την ίδια συνεδρία. Αυτή είναι η **εργαζόμενη μνήμη** — υπάρχει μόνο για τη διάρκεια ζωής της συνεδρίας.

### Τι συμβαίνει με ένα νέο νήμα;

Εάν δημιουργήσουμε μια **νέα** συνεδρία, ο πράκτορας δεν έχει μνήμη της προηγούμενης συνομιλίας:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Πρότυπο Μακροχρόνιας Μνήμης

Για να θυμόμαστε τις προτιμήσεις του χρήστη **σε πολλές συνεδρίες**, χρειαζόμαστε μια επίμονη αποθήκη που να ζει έξω από τον νήμα της συνομιλίας. Ο πράκτορας προσπελαύνει αυτή την αποθήκη μέσω **εργαλείων** — συναρτήσεις που μπορεί να καλεί για να αποθηκεύει και να ανακτά πληροφορίες.

Παρακάτω υλοποιούμε μια απλή αποθήκη προτιμήσεων στη μνήμη (στην παραγωγή θα στηρίζατε αυτό σε μια βάση δεδομένων ή ευρετήριο διανυσμάτων) και τη διαθέτουμε ως εργαλεία που ο πράκτορας μπορεί να χρησιμοποιεί.

### Αρχιτεκτονική
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Σενάριο 1 — Ο χρήστης που μπαίνει για πρώτη φορά κλείνει ταξίδι για επέτειο

Η Σάρα επισκέπτεται για πρώτη φορά. Ο πράκτορας θα πρέπει να αποθηκεύσει τις προτιμήσεις της μέσω των εργαλείων και να τις χρησιμοποιήσει για να προτείνει ξενοδοχεία.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Σενάριο 2 — Η Sarah επιστρέφει εβδομάδες αργότερα

Η Sarah ξεκινά ένα **εντελώς νέο νήμα** (προσομοιώνοντας μια νέα συνεδρία). Η ενεργή μνήμη είναι άδεια, αλλά η αποθήκη προτιμήσεων μακροπρόθεσμα εξακολουθεί να έχει τις πληροφορίες της. Ο πράκτορας πρέπει να τις ανακτήσει και να τις χρησιμοποιήσει για να εξατομικεύσει τις προτάσεις.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Περίληψη

Σε αυτό το μάθημα μάθατε τρεις τύπους μνήμης πράκτορα και πώς να τους υλοποιήσετε με το Microsoft Agent Framework:

| Τύπος Μνήμης | Μηχανισμός MAF | Διάρκεια Ζωής |
|---|---|---|
| **Εργασιακή** | `agent.create_session()` | Μία συνομιλία |
| **Βραχυπρόθεσμη** | Συσσωρευμένο πλαίσιο εντός ενός νήματος | Μία εργασία / συνεδρία |
| **Μακροπρόθεσμη** | Εξωτερικό αποθετήριο προσβάσιμο μέσω λειτουργιών `@tool` | Μεταξύ συνεδριών |

### Βασικά Συμπεράσματα
1. **`agent.create_session()`** παρέχει εργασιακή μνήμη — ο πράκτορας βλέπει ολόκληρο το ιστορικό συνομιλίας μέσα σε μια συνεδρία.
2. **Οι νέες συνεδρίες χάνουν το πλαίσιο** — χωρίς μακροπρόθεσμη μνήμη ο πράκτορας δεν μπορεί να ανακαλέσει παλιές συνομιλίες.
3. **Οι λειτουργίες `@tool`** γεφυρώνουν το κενό — επιτρέπουν στον πράκτορα να αποθηκεύει και να ανακτά πληροφορίες από μόνιμο αποθετήριο.
4. **Η προσωποποίηση βελτιώνεται με τον χρόνο** — όσο περισσότερες προτιμήσεις αποθηκεύονται, τόσο καλύτερες είναι οι συστάσεις του πράκτορα.

### Εφαρμογές στον Πραγματικό Κόσμο
- **Εξυπηρέτηση Πελατών**: Θυμάται το ιστορικό και τις προτιμήσεις πελατών
- **Προσωπικοί Βοηθοί**: Διατηρεί το πλαίσιο σε βάθος ημερών ή εβδομάδων
- **Υγειονομική Περίθαλψη**: Παρακολουθεί πληροφορίες και προτιμήσεις ασθενών
- **Ηλεκτρονικό Εμπόριο**: Προσωποποιημένες αγορές βάσει ιστορικού

### Επόμενα Βήματα
- Αντικατάσταση του λεξικού εντός μνήμης με βάση δεδομένων ή αποθετήριο διανυσμάτων (π.χ. Azure AI Search)
- Προσθήκη λήξης μνήμης για πληροφορίες ευαίσθητες στον χρόνο
- Δημιουργία συστημάτων με πολλούς πράκτορες με κοινή μνήμη
- Εξερεύνηση του σημειωματάριου Cognee για μνήμη υποστηριζόμενη από γράφο γνώσης


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
